In [ ]:
import osmnx as ox
import geopandas as gpd
import matplotlib.pyplot as plt
import os

center_point = (30.6762, 104.0854)  # (lat, lon)

# Distance from center in meters (e.g. 3000 m = 3 km radius)
dist = 6000  

G = ox.graph_from_point(center_point, dist=dist, network_type='drive', simplify=True, retain_all=True)

# Check network size
print(f"Nodes: {len(G.nodes)}")
print(f"Edges: {len(G.edges)}")

# Plot the network
ox.plot_graph(ox.project_graph(G), node_size=0, edge_linewidth=0.5)

In [ ]:
# 2. Simplify the graph (Optional, but often useful for FMM)
#    Simplification removes interstitial nodes (nodes that aren't intersections
#    or dead-ends), potentially speeding up FMM. It keeps the geometry.
#    Check if FMM documentation recommends simplified or original topology.

# Choose which graph to save (simplified or original projected)
# Generally, the simplified one is good for FMM, but check FMM docs.
graph_to_save = G
# graph_to_save = G_proj # Uncomment this line if you want the unsimplified projected graph


# --- Save the Network as Shapefile ---
output_folder = os.path.join(ROOT_DIR, "datasets/osm/cd")

# Create the output directory if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Created output folder: {output_folder}")

print(f"Saving the graph to shapefile in folder: {output_folder}...")
# This function saves nodes and edges into separate shapefiles (nodes.shp, edges.shp)
# within the specified folder path. FMM typically uses the edges.shp file.
#ox.save_graph_shapefile(graph_to_save, filepath=output_folder)

# Convert the graph to GeoDataFrames
gdf_nodes, gdf_edges = ox.convert.graph_to_gdfs(graph_to_save)

# Save the GeoDataFrames as shapefiles
gdf_nodes.to_file(output_folder + '/nodes.shp')
gdf_edges.to_file(output_folder + '/edges.shp')

print("Graph saved successfully!")
print(f"Shapefiles (nodes.shp, edges.shp, etc.) are located in the '{output_folder}' directory.")
print(f"You will likely need '{os.path.join(output_folder, 'edges.shp')}' for FMM.")




In [ ]:
# --- Optional: Plot the Network ---
try:
    print("Plotting the network (may take time for large networks)...")
    fig, ax = ox.plot_graph(graph_to_save, node_size=0, edge_linewidth=0.5, show=False, close=False)
    # Add title with CRS information
    ax.set_title(f"Chengdu Road Network (Simplified, Projected: {graph_to_save.graph['crs']})")
    plt.suptitle(f"Nodes: {len(graph_to_save.nodes)}, Edges: {len(graph_to_save.edges)}", y=0.92) # Add node/edge count below title
    plt.show()
except Exception as e:
    print(f"Could not plot the graph: {e}")

print("Script finished.")

In [4]:

import os
import sys
import geopandas as gpd

sys.path.append("..")
from pipelines.utils import ROOT_DIR

In [5]:
gdf_edges = gpd.read_file(
        os.path.join(ROOT_DIR, f"datasets/osm/cd/edges.shp")
    )

In [7]:
# Add 'fid' column with indices from 0 to the number of rows
gdf_edges['fid'] = range(len(gdf_edges))



In [9]:
# Save the updated GeoDataFrame as a shapefile
gdf_edges.to_file(os.path.join(ROOT_DIR, f"datasets/osm/cd/edges.shp"))